<a href="https://colab.research.google.com/github/wyldescience/FolSum/blob/main/batch%20inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run batch processing on raw images to get counts of folsomia candida hatchlings
- hough transform to crop out background of images
- tile images for better inference (works better for small images and is how the model was trained)
- Obtain raw counts from model and also add column with corrected counts based on linear model with y = -20.5775 + 1.0365


### **Overview**

This notebook runs a trained YOLO object-detection model across a large set of springtail images to produce a single results table containing:

Raw YOLO counts (number of detected springtails per image), and

Bias-corrected counts using a fixed linear correction model (slope + intercept) derived from your validation/calibration work.

It is designed for high-throughput “production” inference (many images), and avoids saving large intermediate files (e.g., tiles) unless you explicitly enable QC outputs.

### **What the Notebook Does**

**(1) Loads your trained YOLO model**

You provide the path to best.pt (your trained weights). The model is loaded once and reused for all images.

**(2) Processes each image using the same steps as validation**

For every input image:

A. Arena cropping (automatic)

The script detects the arena region (circular/large boundary) and crops the image to that arena.

Pixels outside the arena are masked to black to reduce false detections.

B. Tiling (in memory)

The cropped image is split into overlapping tiles (default 640×640 with ~25% overlap).

This helps detection in large images and prevents missing small individuals.

C. YOLO inference on tiles

The YOLO model runs on each tile and outputs bounding boxes + confidence values.

D. Merge duplicate detections across overlapping tiles

Because tiles overlap, a springtail can be detected in more than one tile.

The script merges duplicates using a global Non-Maximum Suppression (NMS) step based on IoU (intersection-over-union).

E. Count detections

The final number of merged bounding boxes is recorded as raw_count.

**(3) Applies a fixed linear correction to raw counts**

Based on your calibration against manual counts, the script computes:
corrected_count = max(0, intercept + slope * raw_count)

**(4) Writes a single output CSV**

A single tidy CSV is saved containing one row per image.

## **Mount drive**

In [ ]:
import os
import shutil

if os.path.exists("/content/drive"):
    shutil.rmtree("/content/drive")

print("Cleared old mountpoint.")

Cleared old mountpoint.


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## **Install packages**

In [ ]:
!pip -q install ultralytics opencv-python

import os, re, glob, random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## **Settings for YOLO model**


In [ ]:
# =========================
# PATHS
# =========================
ROOT = Path("/content/drive/MyDrive/YOLO nymph detector")

# Folder created by your sorting script:
# .../by_generation/F1 ... F8
IMG_ROOT = ROOT / "analysis final data/images/new to be adjusted"   # <-- edit if needed

# Output folder for CSV (and optional QC overlays)
OUT_DIR  = ROOT / "analysis final data/yolo_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# YOLO weights
BEST_WEIGHTS = ROOT / "yolo_runs/nymph_yolov8n_tiles/weights/best.pt"  # <-- edit if needed

# =========================
# CORRECTION MODEL (EDIT)
# =========================
INTERCEPT = -20.58   # <-- put your final intercept here
SLOPE = 1.04         # <-- put your final slope here

def corrected_count(raw):
    return max(0.0, INTERCEPT + SLOPE * float(raw))

# =========================
# YOLO SETTINGS
# =========================
CONF = 0.35
IOU_TILE = 0.6
IOU_GLOBAL = 0.5
MAX_DET = 3000
IMGSZ = 640

# =========================
# CROPPING + TILING
# =========================
TILE_SIZE = 640
OVERLAP = 0.25
PAD = 15

# =========================
# OPTIONAL QC OVERLAYS
# =========================
SAVE_OVERLAYS = True   # True if you want some QC overlays
N_QC_OVERLAYS = 50      # save overlays for up to N random images
QC_DIR = OUT_DIR / "qc_overlays"
QC_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".webp"}

In [ ]:
print("Weights exists?", BEST_WEIGHTS.exists(), BEST_WEIGHTS)
model = YOLO(str(BEST_WEIGHTS))

Weights exists? True /content/drive/MyDrive/YOLO nymph detector/yolo_runs/nymph_yolov8n_tiles/weights/best.pt


## **Helper functions (cropping, tiling)**

In [ ]:
# ---------------- Arena crop (circle-fit) ----------------
def crop_arena_by_contour(img_rgb, pad=15):
    if img_rgb.ndim == 2:
        img_rgb = np.stack([img_rgb]*3, axis=-1)
    if img_rgb.shape[-1] == 4:
        img_rgb = img_rgb[..., :3]
    img_rgb = img_rgb.astype(np.uint8)

    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (9, 9), 0)
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    k1 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k1)
    k2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k2)

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return img_rgb

    H, W = gray.shape
    cx0, cy0 = W / 2, H / 2

    best, best_score = None, -1e18
    for c in cnts:
        area = cv2.contourArea(c)
        if area < 0.05 * H * W:
            continue
        peri = cv2.arcLength(c, True)
        if peri <= 0:
            continue
        circularity = 4 * np.pi * area / (peri * peri)
        (x, y), r = cv2.minEnclosingCircle(c)
        dist = ((x - cx0) ** 2 + (y - cy0) ** 2) ** 0.5
        score = (circularity * 2.0) + (area / (H * W)) - (dist / max(H, W))
        if score > best_score:
            best_score, best = score, c

    if best is None:
        best = max(cnts, key=cv2.contourArea)

    (x, y), r = cv2.minEnclosingCircle(best)
    x, y, r = int(x), int(y), int(r)

    arena_mask = np.zeros((H, W), dtype=np.uint8)
    cv2.circle(arena_mask, (x, y), r, 255, thickness=-1)

    x0 = max(0, x - r - pad); x1 = min(W, x + r + pad)
    y0 = max(0, y - r - pad); y1 = min(H, y + r + pad)

    cropped = img_rgb[y0:y1, x0:x1].copy()
    m = arena_mask[y0:y1, x0:x1] > 0
    cropped[~m] = 0
    return cropped

# ---------------- Tiling ----------------
def tile_image(img_rgb, tile_size=640, overlap=0.25):
    H, W = img_rgb.shape[:2]
    stride = int(tile_size * (1 - overlap))
    stride = max(1, stride)

    tiles = []
    xs = list(range(0, max(1, W - tile_size + 1), stride))
    ys = list(range(0, max(1, H - tile_size + 1), stride))
    if len(xs) == 0: xs = [0]
    if len(ys) == 0: ys = [0]
    if xs[-1] != max(0, W - tile_size): xs.append(max(0, W - tile_size))
    if ys[-1] != max(0, H - tile_size): ys.append(max(0, H - tile_size))

    for y0 in ys:
        for x0 in xs:
            tile = img_rgb[y0:y0+tile_size, x0:x0+tile_size].copy()
            th, tw = tile.shape[:2]
            if th < tile_size or tw < tile_size:
                pad_img = np.zeros((tile_size, tile_size, 3), dtype=np.uint8)
                pad_img[:th, :tw] = tile
                tile = pad_img
            tiles.append((tile, x0, y0))
    return tiles

# ---------------- IoU + NMS ----------------
def iou_xyxy(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    if inter <= 0: return 0.0
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = max(0, b[2]-b[0]) * max(0, b[3]-b[1])
    union = area_a + area_b - inter + 1e-9
    return inter / union

def nms_numpy(boxes, scores, iou_thr=0.5):
    if len(boxes) == 0:
        return []
    idxs = np.argsort(scores)[::-1].tolist()
    keep = []
    while idxs:
        i = idxs.pop(0)
        keep.append(i)
        idxs = [j for j in idxs if iou_xyxy(boxes[i], boxes[j]) < iou_thr]
    return keep

def draw_boxes(img_rgb, boxes, color=(0,255,0), thickness=2):
    out = img_rgb.copy()
    for (x1,y1,x2,y2) in boxes:
        cv2.rectangle(out, (int(x1),int(y1)), (int(x2),int(y2)), color, thickness)
    return out

# ---------------- Parse isoline + generation from filename ----------------
NAME_RE = re.compile(r"^(?P<isoline>I\d+?)_(?P<gen>F\d+?)_.*$", re.IGNORECASE)

def parse_meta(filename):
    m = NAME_RE.match(filename)
    if not m:
        return None, None
    return m.group("isoline").upper(), m.group("gen").upper()

## **Collect image paths**

In [ ]:
# =========================
# SELECT ALL GENERATIONS
# =========================

# img_paths = []
# for g in [f"F{i}" for i in range(1, 9)]:
#     gen_dir = IMG_ROOT / g
#     if not gen_dir.exists():
#         print("Missing gen folder:", gen_dir)
#         continue
#     for p in gen_dir.rglob("*"):
#         if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
#             img_paths.append(p)

# print("Total images found:", len(img_paths))
# print("First few:", img_paths[:5])


# =========================
# SELECT ONE GENERATION
# =========================

GEN = "F1"  # <-- change to F1..F8 when you rerun

gen_dir = IMG_ROOT / GEN
assert gen_dir.exists(), f"Missing folder: {gen_dir}"

img_paths = [p for p in gen_dir.rglob("*")
             if p.is_file() and p.suffix.lower() in IMAGE_EXTS]

print("Generation:", GEN)
print("Images found:", len(img_paths))
print("First few:", img_paths[:5])

Generation: F1
Images found: 764
First few: [PosixPath('/content/drive/MyDrive/YOLO nymph detector/analysis final data/images/new to be adjusted/F1/I10_F1_Y25_swi_R1_MH_31-08-23.JPG'), PosixPath('/content/drive/MyDrive/YOLO nymph detector/analysis final data/images/new to be adjusted/F1/I3_F1_Y25_CON_R2_MH_31-08-23.JPG'), PosixPath('/content/drive/MyDrive/YOLO nymph detector/analysis final data/images/new to be adjusted/F1/I1_F1_Y25_SWI_R3_MH_31-08-23.JPG'), PosixPath('/content/drive/MyDrive/YOLO nymph detector/analysis final data/images/new to be adjusted/F1/I3_F1_Y25_CON_R4_MH_31-08-23.JPG'), PosixPath('/content/drive/MyDrive/YOLO nymph detector/analysis final data/images/new to be adjusted/F1/I1_F1_O20_CON_R5_MH_04-09-23.JPG')]


## **Setup QC overlay sampling**

saves overlays with bounding boxes to check working ok

In [ ]:
qc_set = set(random.sample(img_paths, min(N_QC_OVERLAYS, len(img_paths)))) if SAVE_OVERLAYS else set()
print("QC overlays to save:", len(qc_set))

QC overlays to save: 50


## **Main inference and creation of csv output**

In [ ]:
rows = []

for idx, p in enumerate(img_paths, start=1):
    fn = p.name
    isoline, gen = parse_meta(fn)

    bgr = cv2.imread(str(p))
    if bgr is None:
        rows.append({"image_path": str(p), "filename": fn, "error": "unreadable"})
        continue

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # crop + tile
    cropped = crop_arena_by_contour(rgb, pad=PAD)
    tiles = tile_image(cropped, tile_size=TILE_SIZE, overlap=OVERLAP)

    all_boxes = []
    all_scores = []

    # predict on tiles (in memory)
    for tile, x0, y0 in tiles:
        res = model.predict(
            source=tile,
            imgsz=IMGSZ,
            conf=CONF,
            iou=IOU_TILE,
            max_det=MAX_DET,
            verbose=False
        )[0]

        if res.boxes is None or len(res.boxes) == 0:
            continue

        xyxy = res.boxes.xyxy.cpu().numpy()
        scores = res.boxes.conf.cpu().numpy()

        for (b, s) in zip(xyxy, scores):
            x1, y1, x2, y2 = b
            all_boxes.append([x1 + x0, y1 + y0, x2 + x0, y2 + y0])
            all_scores.append(float(s))

    all_boxes = np.array(all_boxes, dtype=float)
    all_scores = np.array(all_scores, dtype=float)

    if len(all_boxes) == 0:
        final_boxes = np.zeros((0,4), dtype=float)
        mean_conf = np.nan
    else:
        keep = nms_numpy(all_boxes, all_scores, iou_thr=IOU_GLOBAL)
        final_boxes = all_boxes[keep]
        mean_conf = float(np.mean(all_scores[keep])) if len(keep) else np.nan

    raw = int(len(final_boxes))
    corr = corrected_count(raw)

    overlay_path = ""
    if SAVE_OVERLAYS and p in qc_set:
        overlay = draw_boxes(cropped, final_boxes, color=(0,255,0), thickness=2)
        overlay_path = str(QC_DIR / (p.stem + "_overlay.jpg"))
        cv2.imwrite(overlay_path, cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

    rows.append({
        "image_path": str(p),
        "filename": fn,
        "isoline": isoline,
        "generation": gen,
        "raw_count": raw,
        "corrected_count": corr,
        "mean_conf": mean_conf,
        "n_tiles": len(tiles),
        "error": "",
        "overlay_path": overlay_path
    })

    # progress print
    if idx % 100 == 0:
        print(f"Processed {idx}/{len(img_paths)}")

df = pd.DataFrame(rows)

# out_csv = OUT_DIR / "springtail_counts_raw_and_corrected.csv" For all gens

RUN_LABEL = "to adjust"  # change per run to avoid overwriting of same generation from different folder

out_csv = OUT_DIR / f"springtail_counts_{GEN}_{RUN_LABEL}_raw_and_corrected.csv"

df.to_csv(out_csv, index=False)

print("Saved:", out_csv)
df.head()

Processed 100/764
Processed 200/764
Processed 300/764
Processed 400/764
Processed 500/764
Processed 600/764
Processed 700/764
Saved: /content/drive/MyDrive/YOLO nymph detector/analysis final data/yolo_outputs/springtail_counts_F1_to adjust_raw_and_corrected.csv


,image_path,filename,isoline,generation,raw_count,corrected_count,mean_conf,n_tiles,error,overlay_path
0,/content/drive/MyDrive/YOLO nymph detector/ana...,I10_F1_Y25_swi_R1_MH_31-08-23.JPG,I10,F1,221,209.26,0.607016,49,,
1,/content/drive/MyDrive/YOLO nymph detector/ana...,I3_F1_Y25_CON_R2_MH_31-08-23.JPG,I3,F1,206,193.66,0.638160,49,,
2,/content/drive/MyDrive/YOLO nymph detector/ana...,I1_F1_Y25_SWI_R3_MH_31-08-23.JPG,I1,F1,153,138.54,0.628142,49,,
3,/content/drive/MyDrive/YOLO nymph detector/ana...,I3_F1_Y25_CON_R4_MH_31-08-23.JPG,I3,F1,115,99.02,0.598103,49,,
4,/content/drive/MyDrive/YOLO nymph detector/ana...,I1_F1_O20_CON_R5_MH_04-09-23.JPG,I1,F1,209,196.78,0.686889,49,,


**Sanity checks**

In [ ]:
print("Rows:", len(df))
print("Unreadable:", (df["error"] == "unreadable").sum() if "error" in df.columns else 0)
print(df.groupby("generation")["raw_count"].describe())